# Inference Engine — Colab

Runs the C++/CUDA engine end to end on a T4. **No training needed**: the engine
is brought up against randomly initialized weights, because numerical parity does
not care whether the model is any good.

Runtime → Change runtime type → **T4 GPU** before running.

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())

## 1. Clone the repo

In [ ]:
# Which export to run against. Change this for a trained export on Drive:
#   pass 1 (bring-up, random weights) -> '/content/export'
#   pass 2 (after training)           -> '/content/drive/MyDrive/wikitext-gpt/export'
EXPORT = '/content/export'

if EXPORT.startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')

%cd /content
!rm -rf wikitext-gpt
!git clone -q https://github.com/williamclymire-tamu/wikitext-gpt.git

# sanity check
!test -f wikitext-gpt/engine/run.cu && echo 'OK: engine found' || echo 'ERROR: missing engine/run.cu'

## 2. Export random weights

`--random` skips the checkpoint and exports a freshly initialized model, so the
engine can be validated before training finishes. Writes flat fp32 `.bin` files
plus per-stage activation fixtures captured from PyTorch.

In [ ]:
%cd /content/wikitext-gpt
import os
if EXPORT == '/content/export':
    !python train/export_weights.py --random --out {EXPORT}
else:
    print('Using trained export at', EXPORT)
!ls {EXPORT} | head
!ls {EXPORT}/parity

## 3. Validate the wiring on CPU first

`test_cpu` builds with plain g++ — no CUDA, no GPU. It checks the scalar C++
reference against the PyTorch fixtures stage by stage, then checks that
incremental KV-cache decode reproduces a full prefill exactly.

If this fails, the bug is in the forward-pass wiring and no amount of staring at
CUDA will help.

In [ ]:
%cd /content/wikitext-gpt/engine
!make test_cpu
!../tests/test_cpu {EXPORT}

## 4. Build the engine

`HEAD_DIM` is compile-time. It must match `head_dim` in `config.json`
(d_model 256 / n_head 4 = 64). The engine checks this at startup and tells you to
rebuild rather than reading past the head boundary.

In [ ]:
!make engine ARCH=sm_75 HEAD_DIM=64

## 5. Parity — the one that matters

Same prompt PyTorch ran, diffed at every stage. Tolerances loosen deliberately
after the embedding: `--use_fast_math` changes `expf`/`tanhf` and cuBLAS reorders
accumulation, so some drift is expected.

In [ ]:
!./engine parity {EXPORT}

## 6. Decode throughput

In [ ]:
!./engine bench {EXPORT} --tokens 256

## 7. Generation smoke test

With `--random` export, output is gibberish (`[id]` tokens). This proves the
sampling loop and KV cache advance correctly. Real text comes after training.

In [ ]:
!./engine generate {EXPORT} --ids 3,4,5 --tokens 20 --temp 0.8 --top-k 40

## 8. Optional — kernel-level tests + benchmarks

Standalone correctness tests for prefill attention, decode attention, LayerNorm,
GELU, and residual add kernels against PyTorch reference fixtures.

In [ ]:
%cd /content/wikitext-gpt
!pip -q install torch  # gen_fixtures.py needs torch
!python tests/gen_fixtures.py all
%cd /content/wikitext-gpt/engine
!make test_kernels ARCH=sm_75
!../tests/test_kernels attention test ../tests/test_data
!../tests/test_kernels decode    test ../tests/test_decode_data
!../tests/test_kernels attention bench
!../tests/test_kernels decode    bench